# 🚗 Clasificación de Llantas Dañadas — Examen Parcial
## Punto 1: Implementación y Entrenamiento de dos Arquitecturas CNN

**Arquitecturas:**
- **Modelo A:** CNN diseñada desde cero (`TireCNN`)
- **Modelo B:** Transfer Learning con `EfficientNet-B0` preentrenado en ImageNet

**Framework:** PyTorch | **Entorno:** Google Colab (GPU T4)

---
## 0. Instalación de dependencias y descarga del dataset

In [ ]:
# Instalar librerías necesarias
!pip install -q kagglehub torchmetrics torchvision matplotlib seaborn scikit-learn grad-cam

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DESCARGA DEL DATASET VÍA kagglehub
# ─────────────────────────────────────────────────────────────────────────────
# La primera vez que corras esto en Colab, kagglehub pedirá tu
# usuario y API token de Kaggle (https://www.kaggle.com/settings/account).
# Una vez autenticado, el token queda cacheado en la sesión.

import kagglehub, shutil, os

print('⏳ Descargando dataset desde Kaggle...')
ruta_cache   = kagglehub.dataset_download('jehanbhathena/tire-texture-image-recognition')
print(f'📥 Archivos cacheados en: {ruta_cache}')

ruta_destino = '/content/dataset_brutoset_bruto'
if os.path.exists(ruta_destino):
    shutil.rmtree(ruta_destino)
os.makedirs(ruta_destino, exist_ok=True)

print('🚚 Moviendo archivos a la ruta de trabajo local...')
shutil.copytree(ruta_cache, ruta_destino, dirs_exist_ok=True)
print('✅ Dataset preparado en /content/dataset_brutoset_bruto/')

In [ ]:
# Verificar estructura del dataset descargado
import os
from pathlib import Path

base = Path('/content/dataset_bruto')
print(f'Explorando: {base}\n')
for root, dirs, files_list in os.walk(base):
    level = str(root).replace(str(base), '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        subindent = '  ' * (level + 1)
        imgs = [f for f in files_list if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if imgs:
            print(f'{subindent}... {len(imgs)} imágenes')


In [ ]:
# Explorar estructura de carpetas
import os
for root, dirs, files_list in os.walk('/content/dataset_bruto'):
    level = root.replace('/content/dataset_bruto', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files_list[:3]:
            print(f'{subindent}{f}')
        if len(files_list) > 3:
            print(f'{subindent}... y {len(files_list)-3} más')

---
## 1. Imports y configuración global

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from torchmetrics import Accuracy, Precision, Recall, F1Score, AUROC

# ── Reproducibilidad ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# ── Dispositivo ───────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {DEVICE}")

# ── Hiperparámetros globales ──────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 20
LR          = 1e-3
DATA_DIR    = Path('/content/dataset_bruto')   # ajusta si la carpeta raíz difiere
NUM_WORKERS = 2

---
## 2. Análisis exploratorio del dataset (EDA)

In [ ]:
# Detectar automáticamente la carpeta de imágenes con subcarpetas de clase
def find_image_root(base: Path) -> Path:
    """Busca recursivamente la primera carpeta que contenga subcarpetas de clase."""
    for dirpath, dirnames, filenames in os.walk(base):
        # Si hay subcarpetas que contengan imágenes, esta es la raíz
        subdirs = [Path(dirpath) / d for d in dirnames]
        if subdirs and any(
            any(f.suffix.lower() in ['.jpg', '.jpeg', '.png']
                for f in sd.iterdir() if sd.is_dir() or f.is_file())
            for sd in subdirs if sd.is_dir()
        ):
            return Path(dirpath)
    return base

IMAGE_ROOT = find_image_root(DATA_DIR)
print(f"Raíz de imágenes detectada: {IMAGE_ROOT}")

# Contar imágenes por clase
class_counts = {}
for class_dir in sorted(IMAGE_ROOT.iterdir()):
    if class_dir.is_dir():
        imgs = list(class_dir.glob('**/*'))
        imgs = [f for f in imgs if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
        class_counts[class_dir.name] = len(imgs)

print("\nDistribución de clases:")
for cls, count in class_counts.items():
    print(f"  {cls}: {count} imágenes")
print(f"\nTotal: {sum(class_counts.values())} imágenes")
print(f"Clases: {list(class_counts.keys())}")

In [ ]:
# Visualización de distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Barplot
classes = list(class_counts.keys())
counts  = list(class_counts.values())
colors  = ['#2ecc71' if 'good' in c.lower() or 'normal' in c.lower() else '#e74c3c'
           for c in classes]
axes[0].bar(classes, counts, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribución de clases', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Número de imágenes')
for i, v in enumerate(counts):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=classes, autopct='%1.1f%%', colors=colors,
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción de clases', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Ratio de desbalance
if len(counts) == 2:
    ratio = max(counts) / min(counts)
    print(f"\n⚠️  Ratio de desbalance: {ratio:.2f}:1")
    if ratio > 1.5:
        print("   → Se requiere estrategia de mitigación (WeightedRandomSampler o focal loss)")

In [ ]:
# Mostrar imágenes de muestra por clase
fig, axes = plt.subplots(len(classes), 5, figsize=(15, 3 * len(classes)))
if len(classes) == 1:
    axes = [axes]

for row, cls in enumerate(classes):
    cls_dir = IMAGE_ROOT / cls
    imgs = list(cls_dir.glob('**/*.jpg')) + list(cls_dir.glob('**/*.jpeg')) + list(cls_dir.glob('**/*.png'))
    sample = random.sample(imgs, min(5, len(imgs)))
    for col, img_path in enumerate(sample):
        img = Image.open(img_path).convert('RGB')
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(cls, fontsize=12, fontweight='bold', rotation=0,
                                       labelpad=60, va='center')

plt.suptitle('Muestras del dataset por clase', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Preprocesamiento y Data Augmentation

In [ ]:
# Estadísticas de ImageNet (usadas para normalizar ambos modelos)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Transformaciones ──────────────────────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("✅ Transformaciones definidas.")
print(f"   Entrenamiento: {len(train_transforms.transforms)} pasos")
print(f"   Validación:    {len(val_transforms.transforms)} pasos")

In [ ]:
# Cargar dataset completo (sin transforms por ahora para el split)
full_dataset = datasets.ImageFolder(root=str(IMAGE_ROOT))
CLASS_NAMES  = full_dataset.classes
NUM_CLASSES  = len(CLASS_NAMES)

print(f"Clases detectadas: {CLASS_NAMES}")
print(f"Número de clases:  {NUM_CLASSES}")
print(f"Total de imágenes: {len(full_dataset)}")

# ── Split estratificado 70 / 15 / 15 ─────────────────────────────────────────
targets = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, stratify=targets, random_state=SEED)
temp_targets = [targets[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_targets, random_state=SEED)

print(f"\nSplit estratificado:")
print(f"  Train:      {len(train_idx)} ({len(train_idx)/len(full_dataset)*100:.1f}%)")
print(f"  Validación: {len(val_idx)} ({len(val_idx)/len(full_dataset)*100:.1f}%)")
print(f"  Test:       {len(test_idx)} ({len(test_idx)/len(full_dataset)*100:.1f}%)")

In [ ]:
from torch.utils.data import Subset

# Subsets con sus transformaciones
class TransformSubset(torch.utils.data.Dataset):
    """Wrapper para aplicar transforms distintos a cada subset."""
    def __init__(self, dataset, indices, transform=None):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        # img ya es PIL porque ImageFolder sin transform devuelve PIL
        if self.transform:
            img = self.transform(img)
        return img, label

# Recargar sin transform para que devuelva PIL
raw_dataset = datasets.ImageFolder(root=str(IMAGE_ROOT), transform=None)

train_set = TransformSubset(raw_dataset, train_idx, train_transforms)
val_set   = TransformSubset(raw_dataset, val_idx,   val_transforms)
test_set  = TransformSubset(raw_dataset, test_idx,  val_transforms)

# ── WeightedRandomSampler para mitigar desbalance en entrenamiento ────────────
train_targets = [targets[i] for i in train_idx]
class_freq    = Counter(train_targets)
class_weights = {cls: 1.0 / freq for cls, freq in class_freq.items()}
sample_weights = [class_weights[t] for t in train_targets]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print("✅ DataLoaders creados con WeightedRandomSampler.")

---
## 4. Modelo A — CNN desde cero (`TireCNN`)

In [ ]:
class ConvBlock(nn.Module):
    """Bloque conv -> BN -> ReLU -> (opcional) MaxPool."""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class TireCNN(nn.Module):
    """
    CNN diseñada desde cero para clasificación de texturas de llantas.
    Arquitectura: 4 bloques convolucionales + cabezal clasificador.
    Entrada: (B, 3, 224, 224) → Salida: (B, num_classes)
    """
    def __init__(self, num_classes: int = 2, dropout: float = 0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32, pool=True),   # → 112×112
            ConvBlock(32,  64, pool=True),   # →  56×56
            ConvBlock(64, 128, pool=True),   # →  28×28
            ConvBlock(128, 256, pool=True),  # →  14×14
        )
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))  # → 1×1
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_avg_pool(x)
        return self.classifier(x)


# Instanciar y verificar
model_scratch = TireCNN(num_classes=NUM_CLASSES).to(DEVICE)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out   = model_scratch(dummy)
print(f"TireCNN — forma de salida: {out.shape}  ✅")

total_params = sum(p.numel() for p in model_scratch.parameters())
print(f"Parámetros totales: {total_params:,}")

---
## 5. Modelo B — Transfer Learning con EfficientNet-B0

In [ ]:
def build_efficientnet(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    EfficientNet-B0 preentrenado en ImageNet.
    - Backbone congelado en fase 1 (sólo se entrena el cabezal).
    - Fase 2: descongelamos las últimas capas para fine-tuning completo.
    """
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Reemplazar cabezal clasificador
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes),
    )
    return model


model_effnet = build_efficientnet(NUM_CLASSES, freeze_backbone=True).to(DEVICE)
out = model_effnet(dummy)
print(f"EfficientNet-B0 — forma de salida: {out.shape}  ✅")

trainable = sum(p.numel() for p in model_effnet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_effnet.parameters())
print(f"Parámetros entrenables: {trainable:,} / {total:,}  ({trainable/total*100:.1f}%)")

---
## 6. Utilidades de entrenamiento

In [ ]:
def compute_class_weights(targets_list, num_classes, device):
    """Calcula pesos de clase inversamente proporcionales a su frecuencia."""
    freq = Counter(targets_list)
    total = len(targets_list)
    weights = torch.tensor(
        [total / (num_classes * freq[i]) for i in range(num_classes)],
        dtype=torch.float32
    ).to(device)
    return weights


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())
    all_probs  = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    return running_loss / total, correct / total, all_probs, all_labels


def train_model(model, model_name, train_loader, val_loader,
                epochs, lr, device, class_weights=None):
    """
    Loop de entrenamiento completo con:
    - CrossEntropyLoss ponderada
    - AdamW optimizer
    - CosineAnnealingLR scheduler
    - Early stopping (patience=5)
    - Guarda el mejor modelo
    """
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  []}
    best_val_loss = float('inf')
    patience, patience_counter = 5, 0
    best_path = f'/content/{model_name}_best.pth'

    print(f"\n{'='*60}")
    print(f" Entrenando: {model_name}")
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        print(f"Epoch {epoch:02d}/{epochs} | "
              f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | "
              f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  → Early stopping en época {epoch}.")
                break

    # Cargar mejores pesos
    model.load_state_dict(torch.load(best_path, map_location=device))
    print(f"  ✅ Mejor modelo cargado desde {best_path}")
    return model, history


print("✅ Utilidades de entrenamiento listas.")

---
## 7. Entrenamiento — Fase 1 (cabezal de EfficientNet) y CNN desde cero

In [ ]:
# Pesos de clase para la loss
cw = compute_class_weights(train_targets, NUM_CLASSES, DEVICE)
print(f"Pesos de clase: {cw.tolist()}")

# ── Entrenar CNN desde cero ───────────────────────────────────────────────────
model_scratch, history_scratch = train_model(
    model_scratch, 'TireCNN',
    train_loader, val_loader,
    epochs=EPOCHS, lr=LR,
    device=DEVICE, class_weights=cw
)

In [ ]:
# ── Entrenar EfficientNet — Fase 1: sólo cabezal ─────────────────────────────
model_effnet, history_effnet_p1 = train_model(
    model_effnet, 'EfficientNet_phase1',
    train_loader, val_loader,
    epochs=10, lr=1e-3,
    device=DEVICE, class_weights=cw
)

In [ ]:
# ── Fase 2: Fine-tuning completo (descongelar backbone) ───────────────────────
print("Descongelando backbone para fine-tuning...")
for param in model_effnet.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model_effnet.parameters() if p.requires_grad)
print(f"Parámetros entrenables ahora: {trainable:,}")

model_effnet, history_effnet_p2 = train_model(
    model_effnet, 'EfficientNet_phase2',
    train_loader, val_loader,
    epochs=EPOCHS, lr=1e-4,          # LR más bajo para fine-tuning
    device=DEVICE, class_weights=cw
)

# Combinar historiales de EfficientNet
history_effnet = {
    k: history_effnet_p1[k] + history_effnet_p2[k]
    for k in history_effnet_p1
}

---
## 8. Curvas de aprendizaje

In [ ]:
def plot_training_curves(histories: dict, save_path: str = None):
    """
    Dibuja curvas de loss y accuracy para uno o varios modelos.
    `histories`: dict con clave = nombre del modelo, valor = dict de historial.
    """
    n = len(histories)
    fig, axes = plt.subplots(n, 2, figsize=(14, 5 * n))
    if n == 1:
        axes = [axes]

    colors = {'train': '#2980b9', 'val': '#e74c3c'}

    for row, (name, hist) in enumerate(histories.items()):
        epochs_range = range(1, len(hist['train_loss']) + 1)

        # Loss
        axes[row][0].plot(epochs_range, hist['train_loss'],
                          color=colors['train'], linewidth=2, label='Train')
        axes[row][0].plot(epochs_range, hist['val_loss'],
                          color=colors['val'],   linewidth=2, label='Val', linestyle='--')
        axes[row][0].set_title(f'{name} — Loss', fontsize=12, fontweight='bold')
        axes[row][0].set_xlabel('Época'); axes[row][0].set_ylabel('Loss')
        axes[row][0].legend(); axes[row][0].grid(alpha=0.3)

        # Accuracy
        axes[row][1].plot(epochs_range, hist['train_acc'],
                          color=colors['train'], linewidth=2, label='Train')
        axes[row][1].plot(epochs_range, hist['val_acc'],
                          color=colors['val'],   linewidth=2, label='Val', linestyle='--')
        axes[row][1].set_title(f'{name} — Accuracy', fontsize=12, fontweight='bold')
        axes[row][1].set_xlabel('Época'); axes[row][1].set_ylabel('Accuracy')
        axes[row][1].set_ylim(0, 1); axes[row][1].legend()
        axes[row][1].grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_training_curves(
    {'TireCNN (desde cero)': history_scratch,
     'EfficientNet-B0 (TL)': history_effnet},
    save_path='/content/training_curves.png'
)

---
## 9. Evaluación en el conjunto de test

In [ ]:
def full_evaluation(model, loader, device, class_names, model_name):
    """
    Evaluación completa: accuracy, precision, recall, F1, AUC-ROC,
    matriz de confusión y curva ROC.
    """
    criterion = nn.CrossEntropyLoss()
    _, acc, all_probs, all_labels = evaluate(model, loader, criterion, device)

    preds       = all_probs.argmax(dim=1).numpy()
    labels_np   = all_labels.numpy()
    probs_np    = all_probs.numpy()
    n_classes   = len(class_names)

    # ── Reporte de clasificación ──────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  {model_name} — Resultados en TEST")
    print(f"{'='*55}")
    print(classification_report(labels_np, preds, target_names=class_names))

    # AUC-ROC
    if n_classes == 2:
        auc = roc_auc_score(labels_np, probs_np[:, 1])
        print(f"AUC-ROC: {auc:.4f}")
    else:
        auc = roc_auc_score(labels_np, probs_np, multi_class='ovr')
        print(f"AUC-ROC (OvR): {auc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Matriz de confusión ───────────────────────────────────────────────────
    cm = confusion_matrix(labels_np, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=axes[0])
    axes[0].set_title(f'{model_name}\nMatriz de Confusión', fontweight='bold')
    axes[0].set_xlabel('Predicho'); axes[0].set_ylabel('Real')

    # ── Curva ROC (binario) ───────────────────────────────────────────────────
    if n_classes == 2:
        fpr, tpr, _ = roc_curve(labels_np, probs_np[:, 1])
        axes[1].plot(fpr, tpr, color='#e74c3c', lw=2,
                     label=f'ROC curve (AUC = {auc:.3f})')
        axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].set_title(f'{model_name}\nCurva ROC', fontweight='bold')
        axes[1].legend(loc='lower right')
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'/content/eval_{model_name.replace(" ","_")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    return {'accuracy': acc, 'auc': auc, 'preds': preds,
            'labels': labels_np, 'probs': probs_np}


results_scratch = full_evaluation(
    model_scratch, test_loader, DEVICE, CLASS_NAMES, 'TireCNN')

results_effnet  = full_evaluation(
    model_effnet, test_loader, DEVICE, CLASS_NAMES, 'EfficientNet-B0')

---
## 10. Tabla comparativa de modelos

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def summary_row(name, res, class_names):
    labels, preds = res['labels'], res['preds']
    avg = 'binary' if len(class_names) == 2 else 'macro'
    return {
        'Modelo':    name,
        'Accuracy':  f"{res['accuracy']:.4f}",
        'Precision': f"{precision_score(labels, preds, average=avg, zero_division=0):.4f}",
        'Recall':    f"{recall_score(labels, preds, average=avg, zero_division=0):.4f}",
        'F1-Score':  f"{f1_score(labels, preds, average=avg, zero_division=0):.4f}",
        'AUC-ROC':   f"{res['auc']:.4f}",
    }

import pandas as pd
rows = [
    summary_row('TireCNN (desde cero)', results_scratch, CLASS_NAMES),
    summary_row('EfficientNet-B0 (TL)', results_effnet,  CLASS_NAMES),
]
df_comparison = pd.DataFrame(rows).set_index('Modelo')
print("\n📊 Tabla comparativa — Conjunto de Test")
print(df_comparison.to_string())
df_comparison.to_csv('/content/model_comparison.csv')

---
## 11. Guardar modelos finales

In [ ]:
torch.save({
    'model_state_dict': model_scratch.state_dict(),
    'class_names':      CLASS_NAMES,
    'num_classes':      NUM_CLASSES,
    'architecture':     'TireCNN',
}, '/content/TireCNN_final.pth')

torch.save({
    'model_state_dict': model_effnet.state_dict(),
    'class_names':      CLASS_NAMES,
    'num_classes':      NUM_CLASSES,
    'architecture':     'EfficientNet-B0',
}, '/content/EfficientNet_final.pth')

print("✅ Modelos guardados en /content/")
print("   - TireCNN_final.pth")
print("   - EfficientNet_final.pth")

# Descargar desde Colab
from google.colab import files
files.download('/content/TireCNN_final.pth')
files.download('/content/EfficientNet_final.pth')
files.download('/content/model_comparison.csv')
files.download('/content/training_curves.png')

---
## 📝 Notas para el informe

### Decisiones de diseño — TireCNN
- **4 bloques Conv-BN-ReLU-Conv-BN-ReLU-MaxPool** progresivos (32→64→128→256 filtros)
- **Global Average Pooling** en lugar de Flatten directo → reduce parámetros y overfitting
- **Batch Normalization** en cada bloque → estabiliza el entrenamiento
- **Dropout(0.4)** antes de la capa lineal final

### Decisiones de diseño — EfficientNet-B0
- Entrenamiento en **dos fases**: cabezal primero (10 épocas, LR=1e-3), luego fine-tuning completo (LR=1e-4)
- Backbone preentrenado en **ImageNet** → representaciones ricas de texturas sin datos extra
- Cabezal personalizado: Linear(1280→256)→ReLU→Dropout→Linear(256→num_classes)

### Estrategia antidesbalance implementada
- **WeightedRandomSampler** en el DataLoader de entrenamiento
- **CrossEntropyLoss ponderada** con pesos inversamente proporcionales a la frecuencia
- Ambas estrategias operan de forma complementaria

### Próximos pasos
- Punto 2: Estudio de ablación (focal loss vs BCE, augmentation strategies)
- Punto 3: Grad-CAM para interpretabilidad
- Punto 4: Análisis cualitativo de fallos (5 ejemplos mal clasificados)